In [1]:
import sys, logging, time
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd
from src.feature_extraction import (
    compute_aa_composition, compute_kmer_frequencies, compute_onehot_mean,
    compute_physicochemical, compute_blosum62_profile, compute_length_and_moments,
    compute_esm2
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
FB_DIR = DATA_DIR / 'fitness_browser'

In [5]:
labeled_pd = load_labeled_pd(DATA_DIR)
protein_ids = labeled_pd.index.tolist()
sequences = labeled_pd['sequence'].tolist()
id_to_seq = labeled_pd['sequence'].to_dict()

In [3]:
# Complete list of 60 organisms
ALL_ORGS = [
    'acidovorax_3H11', 'azobra', 'Btheta', 'Bifido', 'Brev2', 'BFirm',
    'Burk376', 'Burkholderia_OAS925', 'Bvulgatus_CL09T03C04', 'Caulo',
    'CL21', 'Cup4G11', 'PS', 'Dda3937', 'Ddia6719', 'DdiaME23', 'Dino',
    'DvH', 'Dyella79', 'HerbieS', 'Kang', 'Keio', 'Korea', 'Koxy',
    'Lysobacter_OAE881', 'Magneto', 'Marino', 'Methanococcus_JJ',
    'Methanococcus_S2', 'Miya', 'MR1', 'Mucilaginibacter_YX36', 'MycoTube',
    'Pedo557', 'Phaeo', 'Ponti', 'pseudo1_N1B4', 'pseudo13_GW456_L13',
    'pseudo3_N2E3', 'pseudo5_N2C3_1', 'pseudo6_N2E2', 'psRCH2', 'Putida',
    'PV4', 'RalstoniaBSBF1503', 'RalstoniaGMI1000', 'RalstoniaPSI07',
    'RalstoniaUW163', 'rhodanobacter_10B01', 'rhodanobacter_R12',
    'rhodanobacter_T8', 'RPal_CGA009', 'SB2B', 'Smeli', 'SyringaeB728a',
    'SyringaeB728a_mexBdelta', 'SynE', 'Variovorax_OAS795', 'WCS417',
    'Xantho'
]

In [12]:
# Install if needed:
# !pip install curl_cffi

from curl_cffi.requests import Session
from pathlib import Path

BASE_URL = "https://fit.genomics.lbl.gov"
FB_DIR = DATA_DIR / "fitness_browser"

sess = Session(impersonate="chrome131")

for i, org in enumerate(ALL_ORGS):
    filepath = FB_DIR / org / f"{org}_genome_nt.txt"
    if filepath.exists():
        print(f"[{i+1}/{len(ALL_ORGS)}] {org}: already cached")
        continue
    url = f"{BASE_URL}/cgi-bin/orgSeqs.cgi?orgId={org}&type=nt"
    try:
        r = sess.get(url, timeout=30)
        if r.status_code == 200 and len(r.content) > 100:
            filepath.write_bytes(r.content)
            print(f"[{i+1}/{len(ALL_ORGS)}] {org}: OK ({len(r.content)} bytes)")
        else:
            print(f"[{i+1}/{len(ALL_ORGS)}] {org}: HTTP {r.status_code} (size={len(r.content)})")
    except Exception as e:
        print(f"[{i+1}/{len(ALL_ORGS)}] {org}: ERROR {e}")

[1/60] acidovorax_3H11: HTTP 403 (size=6249)


[2/60] azobra: OK (7655839 bytes)


[3/60] Btheta: OK (6398301 bytes)


[4/60] Bifido: OK (2463074 bytes)


[5/60] Brev2: OK (3318185 bytes)


...

[56/60] SyringaeB728a_mexBdelta: OK (6195270 bytes)


[57/60] SynE: OK (2795973 bytes)


[58/60] Variovorax_OAS795: OK (6193429 bytes)


[59/60] WCS417: OK (6271899 bytes)


[60/60] Xantho: OK (5234531 bytes)


In [14]:
# ─── Genome‑level DNA features ────────────────────────────────────
from src.feature_extraction import compute_genome_dna_features

genome_cache = DATA_DIR / 'features_genome_dna.parquet'
if genome_cache.exists():
    genome_feat = pd.read_parquet(genome_cache)
    log.info("Loaded cached genome DNA features.")
else:
    genome_feat = compute_genome_dna_features(FB_DIR, ALL_ORGS, cache_path=genome_cache)
    log.info("Computed genome DNA features and cached.")

# Broadcast per‑organism features to every protein row
org_col = labeled_pd['organism']
genome_feat_aligned = genome_feat.reindex(org_col, fill_value=0)
genome_feat_aligned.index = labeled_pd.index

# Save as a standard feature parquet — no feature_dfs needed
genome_feat_aligned.to_parquet(DATA_DIR / 'features_genome_dna.parquet')
log.info(f"Genome DNA features saved: {genome_feat_aligned.shape}")

2026-06-25 07:18:10,203 INFO Computed genome DNA features and cached.


2026-06-25 07:18:10,291 INFO Genome DNA features saved: (215051, 18)
